## <center><span style="color:blue">Fla</span><span style="color:gold">sk</span> UI for <span style="color:blue">Gemini</span> Resume Optimizer</span></center>
***
#### Where the <span style="color:gold">Gradio</span> UI was used for the <span style="color:orange">OpenAI</span> / <span style="color:orange">ChatGPT</span> optimizer, the following notebook does the same for our <span style="color:blue">Gemini</span> optimizer.
***
### Step 1: Imports
> #### <span style="color:red">Standard imports as other notebooks, only this time we're importing the functions from the Python file ('<span style="color:firebrick">gemini_flash_functions</span>'), as well as <span style="color:green">Flask</span> and its following libraries</span>:
> ##### 1.) "<u><span style="color:green">Flask</u></span>" - to create the web app object;
> ##### 2.) "<u><span style="color:green">render_template</u></span>" - renders HTML templates (this is a webapp, after all);
> ##### 3.) "<u><span style="color:green">request</u></span>" - deals with the job description inputs that the users copy and paste into the program;
> ##### 4.) "<u><span style="color:green">flash</u></span>" - creates temporary, one-time messages for user interaction;
> ##### 5.) "<u><span style="color:green">redirect</u></span>" - redirects user to an actual URL so they don't have to reload the webapp after every interaction;
> ##### 6.) "<u><span style="color:green">url_for</u></span>" - converts functions to actual URL paths inside the <span style="color:green">Flask</span> application; and 
> ##### 7.) "<u><span style="color:green">session</u></span>" - lets you input multiple job descriptions w/o needing to reload the page.

#### One other added import: '<span style="color:green">tempfile</span>'. '<span style="color:green">Template</span>' is a <span style="color:blue">Python</span> module that creates temporary files and directories. It's used to:
> ##### a.) create an actual temporary Markdown file - the downloaded resume - to the browser; and
> ##### b.) it sets up a temporary path to that file and writes the optimized resume to that file.

In [ ]:
# from flask import Flask, render_template, request, flash, redirect, url_for, send_file # jsonify(?)
# import os
# from dotenv import load_dotenv
# import google.generativeai as genai
# from markdown import markdown
# import tempfile

# # import the functions from 'gemini_flash_functions' python file:
# from gemini_flask_functions import (
#     configure_gemini_api,
#     tailored_resume_function_schema,
#     gemini_initialization_with_function_calling,
#     generate_tailoring_prompt,
#     get_gemini_response_with_function_calling
# )

#### Didn't like the look of the the code above (<span style="color:red"><b>^^^</b></span>). That's a pretty loaded import schedule for '<span style="color:blue">gemini_flask_functions</span>,' and we can import the entire thing with the simple import statement: '<span style="color:green ; font-family:Courier new"><b>from gemini_flask_functions import *</b></span>.'

In [1]:
# Cleaner Imports
from flask import Flask, render_template, request, flash, redirect, url_for, session
import os
from dotenv import load_dotenv
import google.generativeai as genai
from markdown import markdown
import tempfile
import logging  # Import the logging module
from gemini_flask_functions import *


SyntaxError: invalid syntax (3179554723.py, line 1)

In [ ]:
# Load environment variables from our .env file
load_dotenv()

In [2]:
# start Flask
app = Flask(__name__)

NameError: name 'Flask' is not defined

#### When you use an app in <span style="color:green">Flask</span>, it stores your information in cookies for the length of time you're using that app. Because we're dealing with your information and data here, it's best to secure things with a "<span style="color:red">SECRET_KEY</span>" set up. That way, when you launch the app, you have a "secret key" to lock in and protect the data you input into the app.
>##### <b>That key is stored in our environment variables for security's sake. It's being assigned the variable value "<span style="color:green ; font-family:Courier new">app.secret_key</span> for later use. With the app in production, it will draw upon the "<span style="color:firebrick ; font-family:Courier new">FLASK_SECRET_KEY</span>," but for development purposes, we're defaluting that value to "<span style="color:green ; font-family:Courier new">dev</span>."</b>

In [3]:
# Bonus: this also avoids the secret key error at launch!
app.secret_key = os.environ.get("FLASK_SECRET_KEY", "dev")

NameError: name 'os' is not defined

#### If you've ever pointed your garage door opener at your garage, clicked the button, and watched in amazement... <i>as nothing happened</i>, you'll appreciate the fact that Python has a built-in logging function we can call on to give us some help on figuring out why that garage door didn't open.

##### <b><span style="color:firebrick ; font-family:Courier new">"%(asctime)s - %(levelname)s - %(message)s"</span></b> breaks down like this:
>##### <b><span style="color:firebrick ; font-family:Courier new">%(asctiem)s</span></b> == the time the issue occured, <b><span style="color:firebrick ; font-family:Courier new">%(levelname)s</b></span> == how big a deal it is, and <b><span style="color:firebrick ; font-family:Courier new">%(message)s</b></span> == what Python has to say about it.

In [ ]:
# Configure logging
logging.basicConfig(level=logging.DEBUG, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

#### Next step is to set up a directory where we can send the uploaded <span style="color:green">Flask</span> files, ie: your resume.

In [4]:
app.config["UPLOAD_FOLDER"] = "uploads"
os.makedirs(app.config["UPLOAD_FOLDER"], exist_ok=True)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


NameError: name 'app' is not defined

#### Time to set up the function schema. See the '<span style="color:coral">res_optimizer_new_and_improved_gemini</span>' notebook. For now, it's basically you telling the '<span style="color:blue">gemini-2.0-flash</span>' engine how you want your data back.

In [ ]:
# Set up the function schema:
function_schema = {
    "name": "tailor_resume",
    "description": "Make my resume fit the job description so I can get a damn recruiter on the phone.",
    "parameters": {
        "type": "object",
        "properties": {
            "tailored_resume": {
                "type": "string",
                "description": "Your Markdown-formatted resume - for easy editing.",
            },
            "additional_suggestions": {
                "type": "string",
                "description": "Not bad! But here are some ideas to make your resume stronger for this position.",
            },
        },
        "required": ["tailored_resume"]
    }
}


#### Now to give <span style="color:blue">gemini-2.0-flash</span> something to play with (what we're calling here, '<b><span style="color:firebrick ; font-family:Courier new">gemini_model</span></b>').

In [ ]:
# Inialize the Gemini 2.0 Flash Engine
gemini_model = genai.GenerativeModel(
    model_name="gemini-2.0-flash",
    tools=[{"function_declarations": [function_schema]}]
)

#### Add the prompt we engineered earlier, through the trial and error outlined in the '<span style="color:coral">res_optimizer_new_and_improved_gemini</span>' notebook.

In [ ]:
# Throw in the Prompt aimed at the resume and job description
def generate_tailoring_prompt(resume_text, jd_text):
    return f"""
You are a professional resume optimization expert.
Optimize the resume I have provided you to align with the given job description following the guidelines below:

1. Make the resume one page, relevant, keyword optimized, action-driven.
2. Format it cleanly in **Markdown**.
3. Optimize for Applicant Tracking Systems (ATS).
4. End with an "**Some Suggestions To Make Your Resume Stronger**" section if relevant.

Resume:
{resume_text}

Job Description:
{jd_text}
"""

### <span style="color:red">Ruh-roh.</span>
#### <span style="color:green">Flask</span> starts up fine as the webapp, but keeps erroring out on a Markdown filter not being found whenever we try to actually use it.
#### After some homework, it looks like that's because <span style="color:green">Flask</span> uses <span style="color:firebrick">Jinja2</span> for its HTML templating, and <span style="color:firebrick">Jinja2</span> doesn't have any sort of Markdown filter. So, we have to make the conversion in the <span style="color:green">Flask</span> route itself.
> ##### Reviewing the code below, the Markdown <i>was</i> converted to HTML (the '<span style="color:hotpink">tailored_resume_html</span> = <span style="color:blue">markdown</span>(<span style="color:forestgreen">tailored_resume</span>)' line), so the problem may rest in the '<b>templates</b>' directory (in <span style="color:blue">VS Code</span>).
> ##### After running the app in the app in its virtual environment, <span style="color:steelblue">Mac Terminal</span> hit me with some more details, signalling an error in the 'Additional Suggestions' result.

In [4]:
# call the api
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY was not loaded from your environment.")
genai.configure(api_key=GEMINI_API_KEY)

In [5]:
# # app = Flask(__name__)
# # app.secret_key = os.environ.get("SECRET_KEY", "dev-secret-key")  # Change this in production

# from flask import Flask, render_template, request, flash, redirect, url_for, send_file
# import os
# import tempfile
# from dotenv import load_dotenv
# from markdown import markdown

# # Import functions from the functions_for_gemini_flask module
# from functions_for_gemini_flask import (
#     configure_gemini_api,
#     tailored_resume_function_schema,
#     gemini_initialization_with_function_calling,
#     generate_tailoring_prompt,
#     get_gemini_response_with_function_calling
# )

# # Load environment variables
# load_dotenv()

# app = Flask(__name__)
# app.secret_key = os.environ.get("SECRET_KEY", "dev-secret-key")  # Change this in production

def parse_gemini_response(response):
    """
    Parse the Gemini API response to extract the tailored resume and suggestions.
    This is an adaptation of the handle_gemini_response function that returns values
    instead of writing to a file.
    """
    tailored_resume = None
    additional_suggestions = None
    
    if response.candidates:
        first_candidate = response.candidates[0]

        if first_candidate.content and first_candidate.content.parts:
            first_part = first_candidate.content.parts[0]

            if first_part.function_call:
                function_call = first_part.function_call

                if function_call.name == "tailor_resume":
                    arguments = function_call.args
                    tailored_resume = arguments.get("tailored_resume")
                    additional_suggestions = arguments.get("additional_suggestions")
            elif first_part.text:
                # Fallback if function calling didn't work
                tailored_resume = first_part.text
    
    return tailored_resume, additional_suggestions

# Configure Gemini API on app startup
try:
    configure_gemini_api()
    function_schema = tailored_resume_function_schema()
    gemini_model = gemini_initialization_with_function_calling(function_schema)
except ValueError as e:
    print(f"Error during initialization: {e}")
    gemini_model = None

@app.route("/", methods=["GET"])
def index():
    """Home page with form to upload resume and job description"""
    return render_template('index.html')
    
@app.route("/tailor-resume", methods=["GET", "POST"])
def tailor_resume():
    """Process the resume and job description to generate a tailored resume"""
    if not gemini_model:
        flash("API configuration error. Please check your environment variables.")
        return redirect(url_for("index"))
    
    # Get resume and job description from form
    resume_text = request.form.get("resume_text")
    jd_text = request.form.get("job_description")
    
    if not resume_text or not jd_text:
        flash("Please provide both resume and job description.")
        return redirect(url_for("index"))
    
    try:
        # Generate tailored resume
        prompt = generate_tailoring_prompt(resume_text, jd_text)
        response = get_gemini_response_with_function_calling(
            gemini_model, 
            prompt, 
            function_schema
        )
        
        tailored_resume, additional_suggestions = parse_gemini_response(response)
        
        if not tailored_resume:
            flash("Failed to generate tailored resume. Please try again.")
            return redirect(url_for("index"))
            
        # Convert Tailored Resume Markdown to HTML for display
        tailored_resume_html = markdown(tailored_resume)
        additional_suggestions_html = markdown(additional_suggestions)

        # Convert Additional Suggestions Markdow to HTML for display
        return render_template(
            "result.html", 
            tailored_resume_html = tailored_resume_html,
            tailored_resume_md = tailored_resume,
            additional_suggestions_html = additional_suggestions_html
        )
        
    except Exception as e:
        flash(f"An error occurred: {str(e)}")
        return redirect(url_for("index"))

@app.route("/download-resume", methods=["POST"])
def download_resume():
    """Download the tailored resume as a markdown file"""
    tailored_resume = request.form.get("tailored_resume_md")
    
    if not tailored_resume:
        flash("No resume data to download.")
        return redirect(url_for("index"))
    
    # Create a temporary file
    with tempfile.NamedTemporaryFile(delete=False, suffix=".md") as temp:
        temp_path = temp.name
        temp.write(tailored_resume.encode("utf-8"))
    
    # Send the file to the user
    try:
        return send_file(
            temp_path,
            as_attachment = True,
            download_name = "tailored_resume.md",
            mimetype = "text/markdown"
        )
    finally:
        # Clean up the temporary file after sending
        os.unlink(temp_path)

if __name__ == "__main__":
    # Check if running in IPython/Jupyter
    try:
        get_ipython
        # If we're in IPython/Jupyter, don't use the auto-reloader
        print("Flask app is ready. Access it at http://127.0.0.1:5000/")
        print("To run the app, use: app.run()")
    except NameError:
        # Normal Python environment
        app.run(debug = True)

Flask app is ready. Access it at http://127.0.0.1:5000/
To run the app, use: app.run()


In [ ]:
app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
